In [ ]:
import requests

In [ ]:
def gerar_coordenadas():
    try:
        import math
        import random

        # Caixa aproximada da cidade de São Paulo
        lat_min, lat_max = -23.75, -23.45
        lon_min, lon_max = -46.80, -46.45

        origem = (random.uniform(lat_min, lat_max), random.uniform(lon_min, lon_max))

        # Destino a 1-5 km da origem, em direção aleatória
        distancia_km = random.uniform(1, 5)
        angulo = random.uniform(0, 2 * math.pi)
        delta_lat = (distancia_km * math.cos(angulo)) / 111.0
        delta_lon = (distancia_km * math.sin(angulo)) / (111.0 * math.cos(math.radians(origem[0])))
        destino = (origem[0] + delta_lat, origem[1] + delta_lon)

        origem = (round(origem[0], 6), round(origem[1], 6))
        destino = (round(destino[0], 6), round(destino[1], 6))
        return origem, destino
    except (TypeError, ValueError):
        return 400, "Parâmetros inválidos para gerar as coordenadas."
    except Exception:
        return 500, "Erro inesperado ao gerar as coordenadas."

In [ ]:
def validar_coordenada(coordenada):
    try:
        import numbers

        # Precisa ser um par (latitude, longitude)
        if not isinstance(coordenada, (tuple, list)) or len(coordenada) != 2:
            raise TypeError
        lat, lon = coordenada
        for valor in (lat, lon):
            if isinstance(valor, bool) or not isinstance(valor, numbers.Real):
                raise TypeError

        # Intervalos válidos (NaN e infinito também caem aqui)
        if not (-90 <= lat <= 90) or not (-180 <= lon <= 180):
            raise ValueError

        return float(lat), float(lon)
    except TypeError:
        return 400, "Latitude e longitude devem ser um par de números."
    except ValueError:
        return 400, "Coordenada fora do intervalo permitido."
    except Exception:
        return 500, "Erro inesperado ao validar a coordenada."

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    try:
        import math

        lat1, lon1, lat2, lon2 = float(lat1), float(lon1), float(lat2), float(lon2)

        raio_terra = 6371000  # metros
        phi1, phi2 = math.radians(lat1), math.radians(lat2)
        dphi = math.radians(lat2 - lat1)
        dlambda = math.radians(lon2 - lon1)

        a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        return raio_terra * c  # distância em metros
    except (TypeError, ValueError):
        return 400, "Não foi possível calcular a distância: coordenadas inválidas."
    except Exception:
        return 500, "Erro inesperado no cálculo da distância."

In [ ]:
def autenticar():
    try:
        ...
    except requests.exceptions.Timeout:
        return 504, "A API demorou demais para responder."
    except requests.exceptions.ConnectionError:
        return 503, "Falha de conexão com a API."
    except requests.exceptions.HTTPError as e:
        return e.response.status_code, "A API recusou a autenticação."
    except PermissionError:
        return 401, "Token ausente ou inválido."
    except ValueError:
        return 502, "Resposta inválida na autenticação."
    except requests.exceptions.RequestException:
        return 503, "Falha na comunicação com a API."
    except Exception:
        return 500, "Erro inesperado na autenticação."

In [ ]:
# Use o mesmo bloco em buscar_posicoes e buscar_paradas_linha
def consultar_api():
    try:
        ...
    except requests.exceptions.Timeout:
        return 504, "A API demorou demais para responder."
    except requests.exceptions.ConnectionError:
        return 503, "Falha de conexão com a API."
    except requests.exceptions.HTTPError as e:
        return e.response.status_code, "A API retornou um erro."
    except (KeyError, TypeError, ValueError):
        return 502, "A API retornou dados vazios ou inconsistentes."
    except requests.exceptions.RequestException:
        return 503, "Falha na comunicação com a API."
    except Exception:
        return 500, "Erro inesperado na consulta."

In [ ]:
def buscar_linhas_candidatas(cliente, origem, destino, raio=None):
    try:
        # cliente: OlhoVivoClient já criado (Membro 1)
        # origem/destino: (lat, lon); raio: metros (1 a 1000), padrão = RAIO_METROS
        if raio is None:
            raio = globals().get("RAIO_METROS")
        if raio is None:
            raise ValueError
        origem = validar_coordenada(origem)
        if isinstance(origem[1], str):
            return origem  # (400, mensagem) vindo da validação
        destino = validar_coordenada(destino)
        if isinstance(destino[1], str):
            return destino
        if isinstance(raio, bool) or not (1 <= raio <= 1000):
            raise ValueError

        # Estratégia: a API não busca paradas por coordenada. Então pegamos as linhas
        # com algum veículo perto da origem (pré-filtro com o dobro do raio, pois um
        # ônibus pode estar um pouco longe da parada) e depois conferimos as paradas
        # de cada linha com o raio exato, tanto na origem quanto no destino.
        # Usamos o JSON bruto (get_json) porque o Membro 3 espera dicts com cp/np/py/px.
        try:
            posicoes = cliente.get_json("/Posicao")

            linhas_proximas = {}
            for linha in posicoes["l"]:
                for veiculo in linha["vs"]:
                    d = haversine(origem[0], origem[1], float(veiculo["py"]), float(veiculo["px"]))
                    if d <= 2 * raio:
                        linhas_proximas[linha["cl"]] = linha
                        break

            candidatas = []
            for codigo, linha in linhas_proximas.items():
                try:
                    paradas = cliente.get_json("/Parada/BuscarParadasPorLinha", codigoLinha=codigo)
                except RespostaInvalidaError:
                    continue  # ignora só a linha com problema e segue com as demais

                tem_origem = False
                tem_destino = False
                for parada in paradas:
                    lat, lon = float(parada["py"]), float(parada["px"])
                    if haversine(origem[0], origem[1], lat, lon) <= raio:
                        tem_origem = True
                    if haversine(destino[0], destino[1], lat, lon) <= raio:
                        tem_destino = True

                # Só é candidata se tiver parada perto da origem E do destino.
                # A escolha do embarque/desembarque exatos é do Membro 3 (escolher_opcao).
                if tem_origem and tem_destino:
                    candidatas.append({
                        "codigo_linha": codigo,
                        "letreiro": linha.get("c"),
                        "sentido": linha.get("sl"),
                        "paradas": paradas,
                    })
        except AutenticacaoError:
            return 401, "Não foi possível autenticar na API."
        except ApiIndisponivelError:
            return 503, "A API está indisponível no momento."
        except RespostaInvalidaError:
            return 502, "A API retornou dados vazios ou inconsistentes."

        if not candidatas:
            raise LookupError

        return candidatas
    except (KeyError, TypeError):
        return 502, "Dados de linhas ou paradas inconsistentes."
    except ValueError:
        return 400, "Origem, destino ou raio inválido."
    except LookupError:
        return 404, "Nenhuma linha atende à origem e ao destino."
    except Exception:
        return 500, "Erro inesperado na busca de linhas."

In [ ]:
def selecionar_opcao():
    try:
        ...
    except (KeyError, IndexError, TypeError):
        return 502, "Dados das paradas incompletos."
    except ValueError:
        return 404, "Nenhuma linha no sentido da viagem."
    except Exception:
        return 500, "Erro inesperado na seleção da linha."

In [ ]:
def buscar_previsao():
    try:
        ...
    except requests.exceptions.Timeout:
        return 504, "A previsão demorou demais para responder."
    except requests.exceptions.ConnectionError:
        return 503, "Falha de conexão ao buscar a previsão."
    except requests.exceptions.HTTPError as e:
        return e.response.status_code, "Erro da API ao buscar a previsão."
    except (KeyError, TypeError):
        return 502, "Dados de previsão inconsistentes."
    except LookupError:
        return 404, "Previsão de chegada indisponível."
    except ValueError:
        return 502, "Resposta de previsão inválida."
    except requests.exceptions.RequestException:
        return 503, "Falha na comunicação ao buscar a previsão."
    except Exception:
        return 500, "Erro inesperado na previsão."

In [ ]:
def mostrar_resumo():
    try:
        ...
    except (KeyError, TypeError, ValueError):
        return 500, "Não foi possível montar o resumo da viagem."
    except Exception:
        return 500, "Erro inesperado ao mostrar os resultados."

In [ ]:
def gerar_mapa():
    try:
        ...
    except ImportError:
        return 500, "Biblioteca de mapas não instalada."
    except (KeyError, TypeError, ValueError):
        return 500, "Dados inválidos para desenhar o mapa."
    except OSError:
        return 500, "Não foi possível salvar o mapa."
    except Exception:
        return 500, "Erro inesperado ao gerar o mapa."